In [1]:
# root file producer for the 16x16
import os
import ROOT as root
import numpy as np
from array import array
import glob
import re
import pandas as pd
from openpyxl import Workbook
import pytz

Welcome to JupyROOT 6.30/04


In [ ]:
file = root.TFile("/Users/icosivi/Desktop/PRE-SERIE/FBK/PRE-SERIE_FBK_Vendor_16x16_IV.root", "RECREATE")
#file = root.TFile("/Users/icosivi/Desktop/PRE-SERIE/FBK/pippo.root", "RECREATE")
tree = root.TTree("Tree","Tree")

event = array('i', [0])
wafer = array('i', [0])
sensor = array('i', [0])
row = array('i', [0])
column = array('i', [0])
fbk_final_grade = array('i', [0])
fbk_iv_grade = array('i', [0])
fbk_optical_inspection = array('i', [0])

V = root.std.vector("float")()
IBACK = root.std.vector("float")()

tree.Branch("wafer", wafer, 'wafer/I')
tree.Branch("event", event, 'event/I')
tree.Branch("sensor", sensor,'sensor/I')
tree.Branch("row", row, 'row/I')
tree.Branch("column", column, 'column/I')
tree.Branch("fbk_final_grade", fbk_final_grade, 'fbk_final_grade/I')
tree.Branch("fbk_iv_grade", fbk_iv_grade, 'fbk_iv_grade/I')
tree.Branch("fbk_optical_inspection", fbk_optical_inspection, 'fbk_optical_inspection/I')


V.reserve(1000)
IBACK.reserve(1000)
tree.Branch("V", "std::vector<float>", V)
tree.Branch("IBACK", "std::vector<float>", IBACK)
#tree.Branch("I_GR", "std::vector<float>", IBACK)

txt_files = glob.glob("/Users/icosivi/Desktop/PRE-SERIE/FBK/export/16x16/*.txt")

wb = pd.read_excel('/Users/icosivi/Desktop/PRE-SERIE/FBK/FBK_sensor-col-row.xlsx')
fbk_csv = pd.read_excel('/Users/icosivi/Desktop/PRE-SERIE/FBK/FBK_report/PS-Batch_export.xlsx')

counter = 0

for txt in txt_files:
    #print(txt)
    V.clear()
    #IBACK.clear()

    with open(txt,"r") as t:
        lines_list = t.readlines()
        header_v = lines_list[0]
        
        header_v.strip()
        hv = re.split("\s+", header_v)
        
        #for m in hv[6:]:
        #for m in hv[8:]:    
        #    if not m.isspace():
        #        if m:
        #            #print(m)
        #            V.push_back( abs(float(m)) )

        lines = lines_list[1:]
        
        for line in lines:
            line.strip()
            IBACK.clear()
            V.clear()
            ll = re.split( '\s+', line)
            #print(ll[5])
            #if ll[4] == "I_BACK":
            #if ll[5] == "I_GR":
            if ll[5] == "I_BACK[A]":
            #if ll[5] == "I_COL[A]":
                event[0] = counter
                wafer[0] = int( ll[0] )
                column[0] = int( ll[1] )
                row[0] = int( ll[2] )
                #print(str(column[0])+"   "+str(row[0]))
                sensor[0] = int(wb[(wb['Row'] == row[0]) & (wb['Column'] == column[0])]['Sensor'].iloc[0])
                
                if str( fbk_csv[(fbk_csv['WAFER'] == wafer[0]) & (fbk_csv['DEVID'] == sensor[0])]['FINALGRADE'].iloc[0]) == 'GOOD':
                    fbk_final_grade[0] = 0
                else:
                    fbk_final_grade[0] = 1
                    
                if str( fbk_csv[(fbk_csv['WAFER'] == wafer[0]) & (fbk_csv['DEVID'] == sensor[0])]['IVGRADE'].iloc[0]) == 'GOOD':
                    fbk_iv_grade[0] = 0
                else:
                    fbk_iv_grade[0] = 1
                    
                if str( fbk_csv[(fbk_csv['WAFER'] == wafer[0]) & (fbk_csv['DEVID'] == sensor[0])]['OI'].iloc[0]) == 'passed':
                #if str( fbk_csv[(fbk_csv['WAFER'] == wafer[0]) & (fbk_csv['ROW'] == row[0]) & (fbk_csv['COL'] == column[0]) ]['OI'].iloc[0]) == 'passed':
                    fbk_optical_inspection[0] = 0
                else:
                    fbk_optical_inspection[0] = 1
                  
                '''
                if ll[4] == "I_BACK":
                    type_def = re.split("_", ll[3])
                    #print(type_def[0])
                    if(type_def[2]=='PIN'):
                        type[0] = 0
                        event[0] = counter
                        wafer[0] = int( ll[0] )
                        column[0] = int( ll[1] )
                        row[0] = int( ll[2] )
                    elif(type_def[2]=='PAD'):
                        type[0] = 1
                        event[0] = counter
                        wafer[0] = int( ll[0] )
                        column[0] = int( ll[1] )
                        row[0] = int( ll[2] )
                '''
                
                '''
                sensor_types = re.split( '_|-', ll[3])
                ggtype = re.findall(r'\d+',sensor_types[1])
                gr_type[0] = int(ggtype[0])
                for s in sensor_types[1]:
                    if s.isdigit():    
                        gr_type[0] = int( s )
                
                for p in sensor_types[2]:
                    if p.isdigit():    
                        grn[0] = int( p )
                for z in sensor_types[2]:
                    if z.isdigit():    
                        grt[0] = int( z )
                    if z == "S":
                        grt[0] = int(2)
                '''
                i_len = 0
                #for q in ll[7:]:
                for q in ll[6:]:
                    if not q.isspace():
                        if q:
                            IBACK.push_back( abs(float(q)) )
                            i_len+=1
                            #if counter==2:
                                #print(q)          
                #if not any(x == "PIN" for x in sensor_types):
                v_len = 0
                for m in hv[8:]:    
                  if not m.isspace():
                    if m and v_len<i_len:
                      #print(m)
                      V.push_back( abs(float(m)) )
                      v_len+=1
                tree.Fill()
                counter += 1

tree.Write()
file.Write()
file.Close()

In [2]:
def to_binary(number, bit_length):
    """
    Converte un numero decimale in una stringa binaria di lunghezza fissa.
    
    Argomenti:
        number (int): Il numero decimale da convertire.
        bit_length (int): Il numero di bit desiderati in output.
        
    Ritorna:
        str: La rappresentazione binaria invertita (Little Endian) del numero.
    """
    # Converte il numero in binario standard (es: "0b1"), 
    # rimuove il prefisso '0b' e aggiunge zeri a sinistra fino alla lunghezza desiderata
    binary_standard = bin(number)[2:].zfill(bit_length)
    
    # Se il numero originale era più lungo del bit_length, tronchiamo per sicurezza
    binary_fixed = binary_standard[-bit_length:]
    
    # Invertiamo la stringa come richiesto dall'esempio (1 -> 10000 invece di 00001)
    return binary_fixed[::-1]


def to_decimal(binary_str):
    """
    Converte una stringa binaria invertita (Little Endian) nel suo valore decimale.
    Esempio: "10000" -> 1
    """
    # Inverte la stringa per riportare il bit meno significativo (LSB) alla fine
    binary_standard = binary_str[::-1]
    
    # Converte la stringa binaria standard in un intero decimale
    return int(binary_standard, 2)

In [3]:
# producer of the xls to register components on the database for the 16x16
xl_filename="FBK_16x16_PRE-SERIES_GIUSTOOOOOOO"
wb = Workbook()
wws = wb.active
wws["A1"] = "SerialNumber"
wws["B1"] = "Vendor" 
wws["C1"] = "Batch" 
wws["D1"] = "Wafer" 
wws["E1"] = "Geometry"
wws["F1"] = "Row" 
wws["G1"] = "Column"
wws["H1"] = "Sensor Number"

file_qa = root.TFile.Open("/Users/icosivi/Desktop/PRE-SERIE/FBK/PRE-SERIE_FBK_Vendor_16x16_IV.root")
tree_qa = file_qa.Get("Tree")

for j,event in enumerate(tree_qa):
    
    vendor_bit = 1
    bit_2 = 0
    lot_bit = to_binary(1,7)
    wafer_bit = to_binary(event.wafer,5)
    sensor_bit = to_binary(event.sensor,6)
    wws["A%i" %(j+2)] = 'PRE'+str(vendor_bit)+str(bit_2)+str(lot_bit)+str(wafer_bit)+str(sensor_bit)
    
    wws["B%i" %(j+2)] = 'FBK-LFoundry'
    wws["C%i" %(j+2)] = 1
    wws["D%i" %(j+2)] = event.wafer
    wws["E%i" %(j+2)] = '16x16'
    
    #Column, Row = wb.loc[df['Sensor'] == sensor_number, ['Column', 'Row']].values[0]

    wws["F%i" %(j+2)] = event.row
    wws["G%i" %(j+2)] = event.column
    wws["H%i" %(j+2)] = event.sensor

save_path = '/Users/icosivi/Desktop/PRE-SERIE/FBK/'
wb.save(save_path+xl_filename+".xlsx")